# Lab 3 — Multi-Server Agent: Helios's Full Diligence Copilot

This is where the pieces come together. Helios now connects **three** MCP servers at once — the
glossary from Lab 1, a portfolio-economics server wrapping structured deal data, and a
document-room server backed by real PDF retrieval — through `MultiServerMCPClient`. A LangGraph
**supervisor** fans research out across two of those servers in parallel, drafts an Investment
Committee memo, has a QA pass check it, and then pauses for a **human approval** before the memo
can be marked ready for IC. The approval checkpoint is persisted to disk, so it survives a kernel
restart — not just a `MemorySaver` that quietly resets.

## Concepts covered
- `MultiServerMCPClient` — connect to several MCP servers and hand *all* their tools to one agent.
- A **supervisor** graph: `Send`-based fan-out to parallel research nodes, a reducer
  (`operator.add`) to gather their notes, then a linear writer -> QA -> approval pipeline.
- **Human-in-the-loop**: `interrupt()` pauses the graph; `Command(resume=...)` continues it.
- **Checkpointed persistence**: `AsyncSqliteSaver` survives a simulated kernel restart, unlike
  in-memory checkpointing.
- Production notes: Docker per-server, bearer-token auth for remote MCP over HTTP, and what
  changes (nothing, via env vars) to get LangSmith tracing on top of this.

In [1]:
# --- Single setup cell: dependencies, imports, and paths -------------------
%pip install --quiet --break-system-packages \
    "mcp>=1.8.0" "langgraph>=0.2.0" "langgraph-checkpoint-sqlite" "aiosqlite" \
    "langchain>=0.3.0" "langchain-core" "langchain-groq>=0.2.0" "langchain-mcp-adapters>=0.1.0" \
    "langchain-community>=0.3.0" "langchain-chroma>=0.1.4" "chromadb>=0.5.0" "langchain-text-splitters" \
    "python-dotenv>=1.0.1" "pydantic>=2.7.0" "reportlab" "pypdf" "numpy"

import json
import os
import sqlite3
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
SERVERS_DIR = ROOT / "mcp_servers"
DOCS_DIR = DATA_DIR / "diligence_docs"
CHROMA_DIR = DATA_DIR / "chroma_diligence"
PORTFOLIO_DB = DATA_DIR / "solstice_portfolio.db"
CHECKPOINT_DB = DATA_DIR / "helios_checkpoints.sqlite"

for d in (DATA_DIR, SERVERS_DIR, DOCS_DIR):
    d.mkdir(parents=True, exist_ok=True)

GROQ_MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-20b")

print(f"ROOT           = {ROOT}")
print(f"PORTFOLIO_DB   = {PORTFOLIO_DB}")
print(f"DOCS_DIR       = {DOCS_DIR}")
print(f"CHROMA_DIR     = {CHROMA_DIR}")
print(f"GROQ_API_KEY set: {bool(os.getenv('GROQ_API_KEY'))}  (model if set: {GROQ_MODEL})")

Note: you may need to restart the kernel to use updated packages.
ROOT           = /home/claude/build/run/deliver/lab3
PORTFOLIO_DB   = /home/claude/build/run/deliver/lab3/data/solstice_portfolio.db
DOCS_DIR       = /home/claude/build/run/deliver/lab3/data/diligence_docs
CHROMA_DIR     = /home/claude/build/run/deliver/lab3/data/chroma_diligence
GROQ_API_KEY set: False  (model if set: openai/gpt-oss-20b)


## Step 1 — The structured side: a portfolio-economics database

Four tables, denormalized by project name (no joins needed for the lookups Helios does): the
project roster, per-year economics, the risk register, and offtake agreements. This is a
different, richer database than Lab 2's tracker — Lab 2 taught "SQLite over MCP" in general; this
one is the specific data the case study in this notebook runs on.

In [2]:
# --- Portfolio economics database ------------------------------------------
PORTFOLIO_SCHEMA_SQL = """
CREATE TABLE IF NOT EXISTS projects (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    technology TEXT NOT NULL,
    state TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS project_economics (
    id INTEGER PRIMARY KEY,
    project TEXT NOT NULL,
    year INTEGER NOT NULL,
    revenue_musd REAL NOT NULL,
    ebitda_musd REAL NOT NULL
);

CREATE TABLE IF NOT EXISTS risk_register (
    id INTEGER PRIMARY KEY,
    project TEXT NOT NULL,
    risk_description TEXT NOT NULL,
    severity TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS offtake_agreements (
    id INTEGER PRIMARY KEY,
    project TEXT NOT NULL,
    counterparty TEXT NOT NULL,
    structure TEXT NOT NULL,
    price_per_mwh REAL
);
"""

SEED_PROJECTS = [
    ("Cottonwood Solar", "solar", "TX"),
    ("Prairie Horizon Wind", "wind", "KS"),
    ("Sabine River Solar+Storage", "solar+storage", "TX"),
    ("Redtail Ridge Wind", "wind", "OK"),
    ("Blue Mesa Solar", "solar", "NM"),
]

SEED_ECONOMICS = [
    ("Cottonwood Solar", 2024, 18.2, 14.1),
    ("Cottonwood Solar", 2025, 19.4, 14.8),
    ("Prairie Horizon Wind", 2024, 24.6, 19.2),
    ("Prairie Horizon Wind", 2025, 23.9, 18.5),
    ("Sabine River Solar+Storage", 2027, 15.0, 10.5),  # pro forma, partial year (development stage)
    ("Sabine River Solar+Storage", 2028, 21.0, 16.0),  # pro forma, first full year
    ("Redtail Ridge Wind", 2027, 22.0, 17.0),  # pro forma (under construction)
    ("Blue Mesa Solar", 2024, 11.8, 8.1),
    ("Blue Mesa Solar", 2025, 10.5, 6.9),  # merchant exposure since PPA expiry
]

SEED_RISKS = [
    ("Cottonwood Solar", "Two unplanned inverter outages in Q1 2026 caused a temporary output shortfall.", "Medium"),
    ("Prairie Horizon Wind", "PPA expires in 3 years; merchant-tail pricing in SPP has been volatile.", "Medium"),
    ("Sabine River Solar+Storage", "Access-road easement dispute with an adjacent landowner is unresolved.", "High"),
    ("Redtail Ridge Wind", "Turbine supply delay pushed the commercial operation date by one quarter.", "Medium"),
    ("Blue Mesa Solar", "Operating fully merchant since PPA expiry; revenue volatility has increased.", "High"),
]

SEED_OFFTAKE = [
    ("Cottonwood Solar", "Texas municipal aggregator", "fixed", 29.50),
    ("Prairie Horizon Wind", "regional utility", "fixed with escalator", 34.00),
    ("Sabine River Solar+Storage", "corporate offtaker (data center operator)", "fixed, tolling for storage", 31.00),
    ("Redtail Ridge Wind", "regional utility", "fixed", 27.75),
    ("Blue Mesa Solar", "wholesale market (no offtake counterparty)", "merchant", None),
]


def initialize_portfolio_db(db_path: Path = PORTFOLIO_DB) -> Path:
    db_path.parent.mkdir(parents=True, exist_ok=True)
    with sqlite3.connect(db_path) as conn:
        conn.executescript(PORTFOLIO_SCHEMA_SQL)
        if conn.execute("SELECT COUNT(*) FROM projects").fetchone()[0] == 0:
            conn.executemany("INSERT INTO projects(name, technology, state) VALUES (?,?,?)", SEED_PROJECTS)
            conn.executemany(
                "INSERT INTO project_economics(project, year, revenue_musd, ebitda_musd) VALUES (?,?,?,?)",
                SEED_ECONOMICS,
            )
            conn.executemany(
                "INSERT INTO risk_register(project, risk_description, severity) VALUES (?,?,?)", SEED_RISKS
            )
            conn.executemany(
                "INSERT INTO offtake_agreements(project, counterparty, structure, price_per_mwh) VALUES (?,?,?,?)",
                SEED_OFFTAKE,
            )
        conn.commit()
    return db_path


initialize_portfolio_db()
print(f"Portfolio DB ready at {PORTFOLIO_DB}")

Portfolio DB ready at /home/claude/build/run/deliver/lab3/data/solstice_portfolio.db


## Step 2 — The unstructured side: diligence documents

Four short PDFs, generated locally with `reportlab` so this notebook has no dependency on any
real deal's paperwork (or on the network). They stand in for the kind of documents a data room
actually holds: an Independent Engineer excerpt, an environmental/permitting summary, an
interconnection-and-offtake summary, and a portfolio risk narrative.

In [3]:
# --- Generate diligence PDFs locally (no network, no real documents) -------
from reportlab.lib.pagesizes import LETTER
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import inch
from reportlab.platypus import Paragraph, SimpleDocTemplate, Spacer

STYLES = getSampleStyleSheet()


def build_pdf(path: Path, title: str, paragraphs: list[str]) -> None:
    doc = SimpleDocTemplate(str(path), pagesize=LETTER, topMargin=0.9 * inch, bottomMargin=0.9 * inch)
    story = [Paragraph(title, STYLES["Title"]), Spacer(1, 14)]
    for p in paragraphs:
        story.append(Paragraph(p, STYLES["Normal"]))
        story.append(Spacer(1, 10))
    doc.build(story)


DOCS = {
    "cottonwood_solar_ie_report.pdf": (
        "Cottonwood Solar -- Independent Engineer Report (Excerpt)",
        [
            "Cottonwood Solar is a 150 MW single-axis tracking PV facility in ERCOT West Texas, "
            "commercially operational since 2022. Based on twelve months of measured irradiance "
            "and generation data, the Independent Engineer confirms a P50 net capacity factor of "
            "28.4%, consistent with the sponsor's underwriting model within half a percentage point.",
            "The module degradation curve is tracking at 0.5% per year, matching the manufacturer's "
            "warranty schedule. The operations and maintenance contract runs through 2032 with the "
            "original EPC contractor, and covers scheduled inverter maintenance and vegetation "
            "management.",
            "Two unplanned inverter outages occurred in the first quarter of 2026, reducing quarterly "
            "output by approximately 1.2%. Root cause was traced to a firmware defect in a batch of "
            "central inverters; the vendor has issued a patch and the operator has scheduled a "
            "fleet-wide update. The Independent Engineer does not consider this a recurring or "
            "material technical risk, but recommends confirming the patch is fully deployed before "
            "closing.",
            "Overall technical conclusion: performance to date is in line with underwriting "
            "assumptions, with no material design or construction quality concerns identified.",
        ],
    ),
    "sabine_river_environmental_summary.pdf": (
        "Sabine River Solar+Storage -- Environmental & Permitting Summary",
        [
            "Sabine River Solar+Storage is a 100 MW solar / 50 MW-200 MWh storage hybrid project in "
            "ERCOT, currently in late-stage development with a targeted commercial operation date in "
            "2027. Wetlands delineation is complete, and the US Army Corps of Engineers has issued a "
            "Section 404 permit with a minor compensatory mitigation acreage requirement, which the "
            "developer has already satisfied through a mitigation bank credit purchase.",
            "The interconnection agreement is fully executed. Required network upgrades were "
            "finalized at $6.2 million, within the development budget assumed at signing.",
            "One open item: in the fourth quarter of 2025, a boundary survey identified a "
            "discrepancy between the recorded easement and the constructed access road, affecting an "
            "adjacent landowner's parcel. A new survey is underway to resolve the discrepancy. "
            "Counsel considers the litigation risk moderate but notes the item remains unresolved as "
            "of this writing and should be tracked as an open condition to financial close rather "
            "than a closed item.",
            "No other outstanding environmental or permitting exceptions were identified.",
        ],
    ),
    "redtail_ridge_interconnection_offtake.pdf": (
        "Redtail Ridge Wind -- Interconnection & Offtake Summary",
        [
            "Redtail Ridge Wind is a 180 MW wind project in the SPP footprint in Oklahoma, currently "
            "under construction with a revised target commercial operation date of the first quarter "
            "of 2027. The project secured its position in the interconnection queue in 2023; both "
            "the system impact study and the facilities study are complete, and network upgrade "
            "costs are finalized and included in the construction budget.",
            "The turbine supply agreement, signed with a tier-one OEM, was delayed by approximately "
            "six weeks due to a gearbox component shortage affecting the OEM's broader supply chain. "
            "This pushed the previously targeted commercial operation date from the fourth quarter of "
            "2026 to the first quarter of 2027. The construction budget is otherwise tracking "
            "approximately 2% under the approved contingency.",
            "A fifteen-year power purchase agreement is executed with a regional utility offtaker at "
            "a fixed price of $27.75 per MWh, with no escalator. The agreement includes standard "
            "delay liquidated damages provisions that partially offset the schedule slip described "
            "above.",
        ],
    ),
    "blue_mesa_risk_narrative.pdf": (
        "Blue Mesa Solar -- Portfolio Risk Narrative",
        [
            "Blue Mesa Solar is a 120 MW solar facility in New Mexico, commercially operational since "
            "2021. Its original ten-year power purchase agreement expired in 2025, and the project "
            "has since operated as a fully merchant asset, selling into the wholesale market without "
            "a contracted offtake.",
            "Since the PPA expired, merchant revenue has shown increasing quarter-over-quarter "
            "volatility. Two factors are driving this: first, curtailment during high-solar-"
            "penetration midday hours has become more frequent as regional solar buildout has "
            "accelerated; second, basis risk between the project's interconnection node and the "
            "broader regional hub price has widened during several high-demand periods.",
            "Degradation is tracking marginally ahead of the Independent Engineer's original 0.5% "
            "per year curve, which is a secondary contributor to softer output in the most recent "
            "year but is not the primary driver of the revenue variance observed.",
            "Portfolio recommendation: given the combination of full merchant exposure and rising "
            "volatility, the asset management team should evaluate either a shorter-dated financial "
            "hedge or a renewed offtake discussion at the next scheduled asset review, rather than "
            "carrying open merchant exposure indefinitely.",
        ],
    ),
}

for filename, (title, paragraphs) in DOCS.items():
    build_pdf(DOCS_DIR / filename, title, paragraphs)

print(f"Generated {len(DOCS)} PDFs in {DOCS_DIR}:")
for f in sorted(DOCS_DIR.glob("*.pdf")):
    print(" -", f.name)

Generated 4 PDFs in /home/claude/build/run/deliver/lab3/data/diligence_docs:
 - blue_mesa_risk_narrative.pdf
 - cottonwood_solar_ie_report.pdf
 - redtail_ridge_interconnection_offtake.pdf
 - sabine_river_environmental_summary.pdf


## Step 3 — Embeddings, live or offline

RAG needs an embedding function to index and search the PDFs above. On a machine with internet
access, `HuggingFaceEmbeddings` downloads a real sentence-transformer model and gives much better
retrieval quality. This sandbox has no route to huggingface.co, so `get_embeddings()` below tries
the real thing first and falls back to a deterministic local hashing embedder — same
`embed_documents` / `embed_query` interface either way, so nothing downstream (the ingestion cell,
the retriever, `docs_server.py`) needs to know or care which one is active.

In [4]:
# --- Embeddings: real if reachable, deterministic fallback otherwise -------
import hashlib
import re

import numpy as np
from langchain_core.embeddings import Embeddings

EMBEDDING_MODEL_NAME = os.getenv("EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2")


class OfflineHashingEmbeddings(Embeddings):
    """[offline-fallback] Deterministic hashing-trick embedder -- no network, no model download.
    Implements the same Embeddings interface as HuggingFaceEmbeddings so Chroma and the retriever
    code never need to branch on which one is active. Retrieval quality is noticeably lower than
    a real sentence-transformer (no semantics, just hashed token overlap in a fixed-size vector),
    which is an honest tradeoff for zero-network-dependency execution."""

    def __init__(self, dim: int = 384):
        self.dim = dim

    def _embed(self, text: str) -> list[float]:
        vec = np.zeros(self.dim, dtype=float)
        for tok in re.findall(r"[a-z0-9]+", text.lower()):
            h = int(hashlib.md5(tok.encode()).hexdigest(), 16)
            idx = h % self.dim
            sign = 1.0 if (h // self.dim) % 2 == 0 else -1.0
            vec[idx] += sign
        norm = np.linalg.norm(vec)
        return (vec / norm if norm > 0 else vec).tolist()

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return [self._embed(t) for t in texts]

    def embed_query(self, text: str) -> list[float]:
        return self._embed(text)


def get_embeddings():
    """Try a real HuggingFace sentence-transformer; fall back to the offline hasher on any
    failure (no package installed, no network route to huggingface.co, model load error, ...)."""
    try:
        from langchain_huggingface import HuggingFaceEmbeddings

        emb = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)
        emb.embed_query("connectivity check")  # forces the model to actually load
        print(f"Using live HuggingFaceEmbeddings ({EMBEDDING_MODEL_NAME})")
        return emb
    except Exception as e:
        print(f"[offline-fallback] Falling back to OfflineHashingEmbeddings ({type(e).__name__}: {e})")
        return OfflineHashingEmbeddings()


EMBEDDINGS = get_embeddings()

[offline-fallback] Falling back to OfflineHashingEmbeddings (ModuleNotFoundError: No module named 'langchain_huggingface')


In [5]:
# --- Ingest the PDFs into a persisted Chroma store --------------------------
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


def ingest_diligence_docs(docs_dir: Path = DOCS_DIR, persist_dir: Path = CHROMA_DIR) -> Chroma:
    persist_dir.mkdir(parents=True, exist_ok=True)
    splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=120)

    chunks = []
    for pdf_path in sorted(docs_dir.glob("*.pdf")):
        pages = PyPDFLoader(str(pdf_path)).load()
        for page in pages:
            page.metadata["source"] = pdf_path.name
        chunks.extend(splitter.split_documents(pages))

    return Chroma.from_documents(
        chunks,
        embedding=EMBEDDINGS,
        persist_directory=str(persist_dir),
        collection_name="diligence_docs",
    )


vectorstore = ingest_diligence_docs()
print(f"Indexed {vectorstore._collection.count()} chunks into {CHROMA_DIR}")

# Quick sanity check -- does unscoped retrieval surface the right document? The easement dispute
# is specifically Sabine River's issue. With real sentence-transformer embeddings this reliably
# comes back on top; the offline hashing fallback only matches shared vocabulary, not meaning, so
# across just 4 short PDFs it can surface a different project's chunk instead. That's the honest
# limitation flagged above, not a bug in the search call -- and it's exactly why
# `search_documents` (a few cells down) takes an optional `project` filter instead of relying on
# unscoped similarity search alone.
hits = vectorstore.similarity_search("easement dispute with a landowner", k=1)
top_source = hits[0].metadata.get("source")
expected_source = "sabine_river_environmental_summary.pdf"
print(f"Top hit source: {top_source}")
if top_source == expected_source:
    print("Matches the expected document.")
else:
    print(
        f"Does NOT match the expected document ({expected_source}) -- the hashing fallback's "
        "known limitation in action. Project-scoped search below works around this."
    )

/tmp/ipykernel_1520/1875345148.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Indexed 8 chunks into /home/claude/build/run/deliver/lab3/data/chroma_diligence
Top hit source: cottonwood_solar_ie_report.pdf
Does NOT match the expected document (sabine_river_environmental_summary.pdf) -- the hashing fallback's known limitation in action. Project-scoped search below works around this.


## Step 4 — Three MCP servers

Helios connects to three standalone servers:

1. **`glossary_server.py`** — the same diligence glossary from Lab 1 (reused as-is).
2. **`portfolio_server.py`** — wraps `solstice_portfolio.db`: per-project economics, the risk
   register, and offtake terms.
3. **`docs_server.py`** — wraps the diligence documents: list them, read one directly, or
   semantically search across them. Its `search_documents` tool accepts an optional `project`
   filter, which restricts the Chroma search to that project's own document(s) using a metadata
   filter — this matters more than it sounds: with the offline hashing embedder active, unfiltered
   semantic search across only 4 short PDFs is noisy enough to occasionally surface the *wrong*
   project's chunk (a real, honest limitation of a hashing-trick fallback, not a bug in the filter
   itself). Scoping by project when you already know the entity in question is a real production
   RAG pattern, not just a workaround for this lab.

In [6]:
# --- Glossary data for glossary_server.py (same content as Lab 1) ----------
GLOSSARY = [
    {"term": "Power Purchase Agreement (PPA)", "summary": "A long-term contract where a buyer (utility, corporate offtaker, or trading desk) agrees to purchase a project's electricity at a set price or formula for a fixed term, usually 10-20 years. The PPA is the primary revenue certainty lenders and investors underwrite against."},
    {"term": "Capacity Factor", "summary": "Actual energy produced over a period divided by the theoretical maximum if the plant ran at full nameplate capacity the whole time. Solar in the US Southwest often runs 25-32%; wind ranges roughly 30-45% depending on site quality."},
    {"term": "Curtailment", "summary": "When a grid operator or the project itself reduces output below available capacity, usually because of transmission congestion, oversupply, or a plant-level constraint. Curtailment directly reduces revenue and is a key diligence risk item in congested interconnection zones."},
    {"term": "Interconnection Queue", "summary": "The ordered backlog of generation projects waiting for a transmission operator to study and approve their grid connection. Queue position and the resulting network upgrade cost estimate are major schedule and cost risks for development-stage projects."},
    {"term": "Investment Tax Credit (ITC)", "summary": "A US federal tax credit equal to a percentage of a project's eligible capital cost, claimed once the project reaches commercial operation. Bonus adders exist for domestic content, energy communities, and low-income siting."},
    {"term": "Production Tax Credit (PTC)", "summary": "A US federal tax credit paid per MWh generated during a project's first 10 years of operation, historically the default structure for wind. Developers choose ITC or PTC per project based on capital intensity and expected output."},
    {"term": "Debt Service Coverage Ratio (DSCR)", "summary": "Cash available for debt service divided by the debt payment due in a period. Project-finance lenders set a minimum DSCR covenant (often 1.20-1.35x); breaching it can trigger a cash sweep or default."},
    {"term": "Merchant Tail Risk", "summary": "The revenue uncertainty in the years after a project's PPA expires, when it sells into the wholesale market at whatever price prevails. Longer merchant tails generally lower an asset's valuation because future cash flows are less certain."},
    {"term": "Commercial Operation Date (COD)", "summary": "The date a project is certified able to reliably deliver power, triggering PPA revenue, tax credit eligibility, and (for a construction-stage acquisition) the final diligence milestone before funds are released."},
    {"term": "Degradation Rate", "summary": "The annual decline in a solar module's power output over its lifetime, typically 0.4-0.7% per year. Independent Engineers apply a degradation curve when forecasting 20-30 year energy yield for financial models."},
    {"term": "Tax Equity Partnership", "summary": "A financing structure where an investor with US tax liability contributes capital in exchange for the project's tax credits and accelerated depreciation, while the developer retains most of the cash and long-term ownership. Common for both ITC and PTC projects."},
    {"term": "Basis Risk", "summary": "The mismatch between the price at a project's grid node and the hub or index price referenced in its PPA or hedge. Wide or volatile basis erodes the value of an otherwise well-priced offtake contract."},
    {"term": "Independent Engineer (IE) Report", "summary": "A third-party technical due-diligence report covering design, construction quality, resource assessment, and projected performance. Lenders and investors typically require a fresh or updated IE report before closing."},
    {"term": "Notice to Proceed (NTP)", "summary": "The formal milestone authorizing construction to begin, usually gated on financing close, permits in hand, and a signed interconnection agreement. Projects are often bucketed by stage as pre-NTP, under construction, or operating."},
    {"term": "Locational Marginal Price (LMP) / Congestion", "summary": "The real-time or day-ahead price of energy at a specific grid location. LMP can diverge sharply from the broader hub price when transmission congestion limits how much power can flow out of a region -- a key driver of both curtailment and basis risk."},
]

glossary_path = DATA_DIR / "glossary.json"
glossary_path.write_text(json.dumps(GLOSSARY, indent=2), encoding="utf-8")
print(f"Wrote {len(GLOSSARY)} glossary terms to {glossary_path}")

Wrote 15 glossary terms to /home/claude/build/run/deliver/lab3/data/glossary.json


In [7]:
%%writefile mcp_servers/glossary_server.py
"""Helios glossary MCP server -- identical to Lab 1's, reused here as one of three servers."""
from __future__ import annotations

import json
from pathlib import Path

from mcp.server.fastmcp import FastMCP

GLOSSARY_PATH = Path("data/glossary.json")
GLOSSARY: list[dict] = json.loads(GLOSSARY_PATH.read_text(encoding="utf-8"))

mcp = FastMCP("helios-glossary")


def _tokenize(text: str) -> set[str]:
    return {t.lower().strip(".,:;()[]{}") for t in text.replace("_", " ").replace("-", " ").split() if len(t) > 2}


def _score(query: str, text: str) -> int:
    return len(_tokenize(query) & _tokenize(text))


@mcp.tool()
def search_glossary(query: str, top_k: int = 3) -> dict:
    """Search Solstice's renewable-energy diligence glossary and return the top matching terms."""
    ranked = sorted(
        (
            {"term": e["term"], "summary": e["summary"], "score": _score(query, e["term"] + " " + e["summary"])}
            for e in GLOSSARY
        ),
        key=lambda item: item["score"],
        reverse=True,
    )
    matches = ranked[: max(1, top_k)]
    rendered = "\n".join(f"{i}. {m['term']} (score={m['score']}): {m['summary']}" for i, m in enumerate(matches, 1))
    return {"query": query, "top_k": top_k, "matches": matches, "rendered": rendered}


@mcp.resource("glossary://map")
def glossary_map() -> str:
    """Return the full glossary as markdown."""
    lines = ["# Solstice diligence glossary"]
    lines += [f"- **{e['term']}** -- {e['summary']}" for e in GLOSSARY]
    return "\n".join(lines)


@mcp.prompt()
def search_prompt(topic: str, audience: str = "analyst") -> str:
    """Build a retrieval-and-explain prompt template for a given topic and audience."""
    return (
        f"You are briefing a {audience} on {topic} during live diligence. "
        "Look it up in the glossary, then explain in three short parts: what it means, "
        "why it matters for underwriting this deal, and one concrete example."
    )


if __name__ == "__main__":
    mcp.run(transport="stdio")

Writing mcp_servers/glossary_server.py


In [8]:
%%writefile mcp_servers/portfolio_server.py
"""Helios portfolio-economics MCP server -- wraps data/solstice_portfolio.db."""
from __future__ import annotations

import sqlite3
from pathlib import Path

from mcp.server.fastmcp import FastMCP

DB_PATH = Path("data/solstice_portfolio.db")

mcp = FastMCP("helios-portfolio")


def _query(sql: str, params: tuple = ()) -> list[dict]:
    with sqlite3.connect(DB_PATH) as conn:
        conn.row_factory = sqlite3.Row
        return [dict(r) for r in conn.execute(sql, params).fetchall()]


@mcp.resource("portfolio://overview")
def portfolio_overview() -> list[dict]:
    """List every project Solstice is tracking, with technology and state."""
    return _query("SELECT name, technology, state FROM projects ORDER BY name")


@mcp.tool()
def list_projects() -> list[dict]:
    """List every project in the portfolio."""
    return _query("SELECT name, technology, state FROM projects ORDER BY name")


@mcp.tool()
def get_project_economics(project: str) -> list[dict]:
    """Return year-by-year revenue and EBITDA (in $M) for a named project."""
    return _query(
        "SELECT year, revenue_musd, ebitda_musd FROM project_economics WHERE project = ? ORDER BY year", (project,)
    )


@mcp.tool()
def get_risk_register(project: str) -> list[dict]:
    """Return logged risks and their severity for a named project."""
    return _query(
        "SELECT risk_description, severity FROM risk_register WHERE project = ?", (project,)
    )


@mcp.tool()
def get_offtake_agreements(project: str) -> list[dict]:
    """Return offtake / PPA terms for a named project."""
    return _query(
        "SELECT counterparty, structure, price_per_mwh FROM offtake_agreements WHERE project = ?", (project,)
    )


@mcp.prompt()
def portfolio_prompt(project: str, question: str) -> str:
    """Build a structured-lookup prompt for a portfolio question about one project."""
    return (
        f"Answer this question about {project} using only the portfolio tools: {question}. "
        "Pull economics, risk register, and offtake terms as needed, and note which tool each "
        "fact came from."
    )


if __name__ == "__main__":
    mcp.run(transport="stdio")

Writing mcp_servers/portfolio_server.py


In [9]:
%%writefile mcp_servers/docs_server.py
"""Helios document-room MCP server -- wraps data/diligence_docs and its Chroma index.

A separate process can't reach into the notebook's EMBEDDINGS object, so this file carries its
own copy of OfflineHashingEmbeddings. As long as the dim and hashing logic match what built the
index, embeddings computed here are consistent with the ones already stored in Chroma.
"""
from __future__ import annotations

import hashlib
import os
import re
from pathlib import Path

import numpy as np
from langchain_chroma import Chroma
from langchain_core.embeddings import Embeddings
from mcp.server.fastmcp import FastMCP
from pypdf import PdfReader

DOCS_DIR = Path("data/diligence_docs")
CHROMA_DIR = Path("data/chroma_diligence")

# One PDF per project in this lab's dataset -- lets search_documents scope retrieval to a single
# project's own document instead of searching (and sometimes mismatching) across all of them.
PROJECT_TO_FILE = {
    "Cottonwood Solar": "cottonwood_solar_ie_report.pdf",
    "Sabine River Solar+Storage": "sabine_river_environmental_summary.pdf",
    "Redtail Ridge Wind": "redtail_ridge_interconnection_offtake.pdf",
    "Blue Mesa Solar": "blue_mesa_risk_narrative.pdf",
}


class OfflineHashingEmbeddings(Embeddings):
    """[offline-fallback] Must match the notebook's version exactly -- same dim, same hash."""

    def __init__(self, dim: int = 384):
        self.dim = dim

    def _embed(self, text: str) -> list[float]:
        vec = np.zeros(self.dim, dtype=float)
        for tok in re.findall(r"[a-z0-9]+", text.lower()):
            h = int(hashlib.md5(tok.encode()).hexdigest(), 16)
            idx = h % self.dim
            sign = 1.0 if (h // self.dim) % 2 == 0 else -1.0
            vec[idx] += sign
        norm = np.linalg.norm(vec)
        return (vec / norm if norm > 0 else vec).tolist()

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return [self._embed(t) for t in texts]

    def embed_query(self, text: str) -> list[float]:
        return self._embed(text)


def _get_embeddings():
    try:
        from langchain_huggingface import HuggingFaceEmbeddings

        model_name = os.getenv("EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2")
        emb = HuggingFaceEmbeddings(model_name=model_name)
        emb.embed_query("connectivity check")
        return emb
    except Exception:
        return OfflineHashingEmbeddings()


_VECTORSTORE = Chroma(
    persist_directory=str(CHROMA_DIR),
    embedding_function=_get_embeddings(),
    collection_name="diligence_docs",
)

mcp = FastMCP("helios-docs")


@mcp.resource("docs://tree")
def docs_tree() -> list[str]:
    """List every diligence document on file."""
    return sorted(p.name for p in DOCS_DIR.glob("*.pdf"))


@mcp.tool()
def list_documents() -> list[str]:
    """List every diligence document available to search or read."""
    return sorted(p.name for p in DOCS_DIR.glob("*.pdf"))


@mcp.tool()
def read_document_text(filename: str, max_chars: int = 4000) -> str:
    """Read the raw extracted text of one diligence document by filename."""
    path = DOCS_DIR / filename
    if not path.resolve().is_relative_to(DOCS_DIR.resolve()):
        raise ValueError("Path escapes the document room.")
    if not path.exists():
        raise ValueError(f"No such document: {filename}")
    reader = PdfReader(str(path))
    text = "\n".join(page.extract_text() or "" for page in reader.pages)
    return text[:max_chars]


@mcp.tool()
def search_documents(query: str, project: str = "", top_k: int = 4) -> dict:
    """Semantically search the diligence documents. If `project` is given and known, the search
    is scoped to that project's own document via a metadata filter instead of searching
    everything -- more reliable than unscoped semantic search over a handful of short PDFs."""
    filter_ = None
    if project and project in PROJECT_TO_FILE:
        filter_ = {"source": PROJECT_TO_FILE[project]}

    docs = _VECTORSTORE.similarity_search(query, k=top_k, filter=filter_)
    if not docs:
        return {"query": query, "project": project, "matches": [], "rendered": "No relevant passages found."}

    matches = [{"source": d.metadata.get("source", "?"), "excerpt": d.page_content} for d in docs]
    rendered = "\n\n".join(f"[{m['source']}] {m['excerpt']}" for m in matches)
    return {"query": query, "project": project, "matches": matches, "rendered": rendered}


@mcp.prompt()
def explain_document_prompt(filename: str, audience: str = "analyst") -> str:
    """Build a prompt asking for a plain-language explanation of one document."""
    return f"Explain {filename} to a {audience}: what it covers, what it concludes, and any open items."


if __name__ == "__main__":
    mcp.run(transport="stdio")

Writing mcp_servers/docs_server.py


## Step 5 — Connect all three with `MultiServerMCPClient`

In Lab 1, wiring one MCP server into an agent meant hand-rolling a `stdio_client` /
`ClientSession` pair inside a LangChain `@tool`. That's fine for one server; it gets tedious fast
for three. `MultiServerMCPClient` takes a config dict (one entry per server) and returns every
tool from every server as ready-to-use LangChain tools in one call.

In [10]:
# --- Connect to all three servers at once -----------------------------------
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_client = MultiServerMCPClient(
    {
        "glossary": {
            "command": "python3",
            "args": [str(SERVERS_DIR / "glossary_server.py")],
            "transport": "stdio",
            "cwd": str(ROOT),
        },
        "portfolio": {
            "command": "python3",
            "args": [str(SERVERS_DIR / "portfolio_server.py")],
            "transport": "stdio",
            "cwd": str(ROOT),
        },
        "docs": {
            "command": "python3",
            "args": [str(SERVERS_DIR / "docs_server.py")],
            "transport": "stdio",
            "cwd": str(ROOT),
        },
    }
)

all_tools = await mcp_client.get_tools()
tools_by_name = {t.name: t for t in all_tools}
print("Tools from all three servers:", sorted(tools_by_name))

# Smoke-test one tool per server before trusting any of them in the graph below.
print("\nglossary  ->", (await tools_by_name["search_glossary"].ainvoke({"query": "PPA", "top_k": 1}))[:80], "...")
print("portfolio ->", await tools_by_name["list_projects"].ainvoke({}))
print(
    "docs      ->",
    (await tools_by_name["search_documents"].ainvoke({"query": "easement", "project": "Sabine River Solar+Storage"}))[
        :120
    ],
    "...",
)

Tools from all three servers: ['get_offtake_agreements', 'get_project_economics', 'get_risk_register', 'list_documents', 'list_projects', 'read_document_text', 'search_documents', 'search_glossary']



glossary  -> [{'type': 'text', 'text': '{\n  "query": "PPA",\n  "top_k": 1,\n  "matches": [\n    {\n      "term": "Power Purchase Agreement (PPA)",\n      "summary": "A long-term contract where a buyer (utility, corporate offtaker, or trading desk) agrees to purchase a project\'s electricity at a set price or formula for a fixed term, usually 10-20 years. The PPA is the primary revenue certainty lenders and investors underwrite against.",\n      "score": 1\n    }\n  ],\n  "rendered": "1. Power Purchase Agreement (PPA) (score=1): A long-term contract where a buyer (utility, corporate offtaker, or trading desk) agrees to purchase a project\'s electricity at a set price or formula for a fixed term, usually 10-20 years. The PPA is the primary revenue certainty lenders and investors underwrite against."\n}', 'id': 'lc_d5179d14-912e-48cb-a13c-9450c0c21622'}] ...


portfolio -> [{'type': 'text', 'text': '{\n  "name": "Blue Mesa Solar",\n  "technology": "solar",\n  "state": "NM"\n}', 'id': 'lc_57e5d3f1-8eb4-45d6-af07-3a44b7b8d2cd'}, {'type': 'text', 'text': '{\n  "name": "Cottonwood Solar",\n  "technology": "solar",\n  "state": "TX"\n}', 'id': 'lc_95c2a853-abaf-4ff1-a114-3b9b3be62757'}, {'type': 'text', 'text': '{\n  "name": "Prairie Horizon Wind",\n  "technology": "wind",\n  "state": "KS"\n}', 'id': 'lc_c2e24f0d-6e19-4aa3-a60a-010ad6049117'}, {'type': 'text', 'text': '{\n  "name": "Redtail Ridge Wind",\n  "technology": "wind",\n  "state": "OK"\n}', 'id': 'lc_8d377690-e3b2-41ee-9a76-858ea4e040c5'}, {'type': 'text', 'text': '{\n  "name": "Sabine River Solar+Storage",\n  "technology": "solar+storage",\n  "state": "TX"\n}', 'id': 'lc_49b53eb0-b946-4fae-a2dd-74fa8e5c3b3a'}]


docs      -> [{'type': 'text', 'text': '{\n  "query": "easement",\n  "project": "Sabine River Solar+Storage",\n  "matches": [\n    {\n      "source": "sabine_river_environmental_summary.pdf",\n      "excerpt": "One open item: in the fourth quarter of 2025, a boundary survey identified a discrepancy between the\\nrecorded easement and the constructed access road, affecting an adjacent landowner\'s parcel. A new\\nsurvey is underway to resolve the discrepancy. Counsel considers the litigation risk moderate but notes\\nthe item remains unresolved as of this writing and should be tracked as an open condition to financial\\nclose rather than a closed item.\\nNo other outstanding environmental or permitting exceptions were identified."\n    },\n    {\n      "source": "sabine_river_environmental_summary.pdf",\n      "excerpt": "Sabine River Solar+Storage -- Environmental &\\n Permitting Summary\\nSabine River Solar+Storage is a 100 MW solar / 50 MW-200 MWh storage hybrid project in ERCOT,\\nc

## Step 6 — The supervisor graph

The graph: `fanout` sends the question to `pdf_researcher` and `portfolio_researcher` in
parallel (`Send`), `gather` joins them (a reducer, `operator.add`, merges both nodes' notes into
one list without either overwriting the other), `writer` drafts an Investment Committee memo,
`qa` checks it against the notes, and `approval` pauses the graph for a human decision before
`finalize` marks the memo done.

One serialization detail matters here: tools obtained from `MultiServerMCPClient` always return a
**list of content blocks** — one block per item if the underlying MCP tool returned a bare list
(as seen with `list_projects` above), or a single block otherwise. Two small helpers below handle
that consistently everywhere it comes up.

In [11]:
# --- Groq (or offline fallback) + helpers for MultiServerMCPClient results -
def get_groq_llm(temperature: float = 0.0):
    api_key = os.getenv("GROQ_API_KEY")
    if not api_key:
        return None
    try:
        from langchain_groq import ChatGroq

        return ChatGroq(model=GROQ_MODEL, temperature=temperature, api_key=api_key)
    except Exception as e:
        print(f"[offline-fallback] Could not initialize Groq ({e}); falling back.")
        return None


def mcp_result_to_text(result: list[dict]) -> str:
    """Join every content block's text -- use when you just want something printable."""
    return "\n".join(block.get("text", "") for block in result)


def mcp_result_to_objects(result: list[dict]) -> list:
    """Parse each content block's text as JSON -- use when you want the structured data back."""
    objects = []
    for block in result:
        try:
            objects.append(json.loads(block.get("text", "")))
        except json.JSONDecodeError:
            objects.append(block.get("text", ""))
    return objects


LLM = get_groq_llm()
print("Live Groq available" if LLM is not None else "[offline-fallback] No GROQ_API_KEY -- research/writer/QA nodes will use their offline paths")

[offline-fallback] No GROQ_API_KEY -- research/writer/QA nodes will use their offline paths


In [12]:
# --- Supervisor state + the two parallel research nodes --------------------
import operator
from typing import Annotated, TypedDict

from langgraph.types import Send


class SupervisorState(TypedDict, total=False):
    question: str
    project: str
    research_notes: Annotated[list[str], operator.add]  # reducer: both researchers append, neither overwrites
    draft: str
    qa_notes: str
    approved: bool
    final: str


def fanout(state: SupervisorState) -> list[Send]:
    payload = {"question": state["question"], "project": state["project"]}
    return [Send("pdf_researcher", payload), Send("portfolio_researcher", payload)]


async def pdf_researcher(state: SupervisorState) -> dict:
    """RAG over the diligence PDFs, scoped to this project's own document."""
    project, question = state["project"], state["question"]

    raw = await tools_by_name["search_documents"].ainvoke({"query": question, "project": project, "top_k": 3})
    payload = mcp_result_to_objects(raw)[0]
    matches = payload.get("matches", [])

    if not matches:
        return {"research_notes": [f"[pdf] No relevant passages found for {project}."]}

    context = "\n\n".join(f"[{m['source']}] {m['excerpt']}" for m in matches)

    if LLM is not None:
        try:
            prompt = (
                "You are a diligence analyst. Read the excerpts below and extract 3-5 short, "
                "factual bullet notes that help answer the question. One fact per line, no "
                f"preamble, no numbering.\n\nQuestion: {question}\n\nExcerpts:\n{context}"
            )
            response = LLM.invoke(prompt)
            notes = [line.strip("-* ").strip() for line in response.content.splitlines() if line.strip()]
        except Exception as e:
            print(f"[offline-fallback] pdf_researcher Groq call failed ({e}); using raw excerpts.")
            notes = []
    else:
        notes = []

    if not notes:
        # [offline-fallback] Skip LLM bullet extraction -- use the retrieved excerpts directly.
        notes = [m["excerpt"][:240].rsplit(" ", 1)[0] + "..." for m in matches]

    return {"research_notes": [f"[pdf] {n}" for n in notes]}


async def portfolio_researcher(state: SupervisorState) -> dict:
    """Structured lookups against the portfolio-economics database, via the MCP portfolio tools."""
    project = state["project"]

    econ = mcp_result_to_objects(await tools_by_name["get_project_economics"].ainvoke({"project": project}))
    risks = mcp_result_to_objects(await tools_by_name["get_risk_register"].ainvoke({"project": project}))
    offtake = mcp_result_to_objects(await tools_by_name["get_offtake_agreements"].ainvoke({"project": project}))

    notes: list[str] = []
    for prev, curr in zip(econ, econ[1:]):
        growth = ((curr["revenue_musd"] - prev["revenue_musd"]) / prev["revenue_musd"] * 100) if prev["revenue_musd"] else 0.0
        notes.append(f"{curr['year']}: revenue ${curr['revenue_musd']}M (YoY {growth:+.1f}%), EBITDA ${curr['ebitda_musd']}M")
    if econ and not notes:
        r = econ[0]
        notes.append(f"{r['year']}: revenue ${r['revenue_musd']}M, EBITDA ${r['ebitda_musd']}M")

    for r in risks:
        notes.append(f"Risk ({r['severity']}): {r['risk_description']}")

    for o in offtake:
        price = f"${o['price_per_mwh']}/MWh" if o.get("price_per_mwh") is not None else "no fixed price"
        notes.append(f"Offtake: {o['counterparty']} -- {o['structure']} ({price})")

    if not notes:
        notes = [f"No structured records found for {project}."]

    return {"research_notes": [f"[portfolio] {n}" for n in notes]}


def gather(state: SupervisorState) -> dict:
    return {}  # fan-in join point; the operator.add reducer already merged both branches' notes

In [13]:
# --- Writer and QA nodes -----------------------------------------------------
import re

NUMBER_RE = re.compile(r"\$?\d[\d,]*\.?\d*%?")


def writer_node(state: SupervisorState) -> dict:
    notes = state.get("research_notes", [])
    question, project = state["question"], state["project"]
    notes_block = "\n".join(f"- {n}" for n in notes)

    if LLM is not None:
        try:
            prompt = (
                "You are a diligence analyst at Solstice Capital Partners. Using only the research "
                f"notes below, write a concise Investment Committee memo (3-4 short paragraphs) for "
                f"{project} that answers the question.\n\nQuestion: {question}\n\nResearch notes:\n{notes_block}"
            )
            draft = f"# Investment Committee Memo -- {project}\n\n" + LLM.invoke(prompt).content
            return {"draft": draft}
        except Exception as e:
            print(f"[offline-fallback] writer Groq call failed ({e}); using a templated draft.")

    # [offline-fallback] Deterministic templated memo, built directly from the same notes --
    # same question-in / memo-out contract, no LLM prose.
    pdf_notes = [n[len("[pdf] "):] for n in notes if n.startswith("[pdf]")]
    portfolio_notes = [n[len("[portfolio] "):] for n in notes if n.startswith("[portfolio]")]
    draft = (
        f"# Investment Committee Memo -- {project}\n\n"
        f"**Question:** {question}\n\n"
        "## Document findings\n" + "\n".join(f"- {n}" for n in pdf_notes) + "\n\n"
        "## Portfolio data\n" + "\n".join(f"- {n}" for n in portfolio_notes)
    )
    return {"draft": draft}


def qa_node(state: SupervisorState) -> dict:
    draft = state.get("draft", "")
    notes = state.get("research_notes", [])
    notes_block = "\n".join(notes)

    # Deterministic check that runs regardless of LLM availability: every number quoted in the
    # draft should trace back to a number that actually appears in the research notes. A number
    # that doesn't is a concrete, checkable sign of a possibly unsupported figure.
    draft_numbers = {n for n in NUMBER_RE.findall(draft) if len(n) > 1}
    notes_numbers = {n for n in NUMBER_RE.findall(notes_block) if len(n) > 1}
    unsupported = sorted(draft_numbers - notes_numbers)
    deterministic_finding = (
        f"Unsupported figures found in the draft (not traceable to a research note): {', '.join(unsupported)}"
        if unsupported
        else "Every number in the draft traces back to a research note."
    )

    if LLM is not None:
        try:
            prompt = (
                "You are a meticulous diligence QA reviewer. Check the draft memo against the "
                "research notes for factual consistency, missing information, and unsupported "
                "claims. Reply with a short bullet list of findings, or exactly 'No major issues "
                f"found.' if it checks out.\n\nResearch notes:\n{notes_block}\n\nDraft memo:\n{draft}"
            )
            qa_notes = LLM.invoke(prompt).content.strip()
            return {"qa_notes": f"{qa_notes}\n\n(Deterministic numeric check: {deterministic_finding})"}
        except Exception as e:
            print(f"[offline-fallback] qa Groq call failed ({e}); using the deterministic check only.")

    return {"qa_notes": f"[offline-fallback] {deterministic_finding}"}

In [14]:
# --- Human approval gate -----------------------------------------------------
from langgraph.types import interrupt


def approval_gate(state: SupervisorState) -> dict:
    decision = interrupt(
        {
            "title": "Investment Committee sign-off required",
            "project": state["project"],
            "draft": state["draft"],
            "qa_notes": state.get("qa_notes", ""),
            "allowed": ["approve", "reject"],
        }
    )
    approved = False
    if isinstance(decision, dict):
        approved = str(decision.get("decision", "")).lower() == "approve" or bool(decision.get("approved"))
    elif isinstance(decision, str):
        approved = decision.strip().lower() in {"approve", "yes", "y", "true"}
    return {"approved": approved}


def finalize(state: SupervisorState) -> dict:
    if state.get("approved"):
        return {
            "final": state["draft"]
            + "\n\n## QA Review\n"
            + state.get("qa_notes", "")
            + "\n\n**Status: Approved for Investment Committee.**"
        }
    return {"final": "Memo drafted but not approved -- returned to the deal team for further work."}

In [15]:
# --- Build the supervisor graph ----------------------------------------------
from langgraph.constants import END, START
from langgraph.graph import StateGraph


def build_supervisor_graph(checkpointer):
    builder = StateGraph(SupervisorState)

    builder.add_node("fanout", lambda s: s)
    builder.add_node("pdf_researcher", pdf_researcher)
    builder.add_node("portfolio_researcher", portfolio_researcher)
    builder.add_node("gather", gather)
    builder.add_node("writer", writer_node)
    builder.add_node("qa", qa_node)
    builder.add_node("approval", approval_gate)
    builder.add_node("finalize", finalize)

    builder.add_edge(START, "fanout")
    builder.add_conditional_edges("fanout", fanout, ["pdf_researcher", "portfolio_researcher"])
    builder.add_edge("pdf_researcher", "gather")
    builder.add_edge("portfolio_researcher", "gather")
    builder.add_edge("gather", "writer")
    builder.add_edge("writer", "qa")
    builder.add_edge("qa", "approval")
    builder.add_edge("approval", "finalize")
    builder.add_edge("finalize", END)

    return builder.compile(checkpointer=checkpointer)


print("build_supervisor_graph ready")

build_supervisor_graph ready


In [16]:
# --- Demo run 1: Cottonwood Solar -- research, draft, QA, then approve ------
from langgraph.checkpoint.memory import MemorySaver

graph = build_supervisor_graph(MemorySaver())
config_1 = {"configurable": {"thread_id": "cottonwood-2026-07"}}

result_1 = await graph.ainvoke(
    {
        "question": "How is the project performing against underwriting, and are there any open technical concerns?",
        "project": "Cottonwood Solar",
    },
    config_1,
)

print(result_1["draft"])
print("\n--- QA review ---")
print(result_1["qa_notes"])
print("\n--- Paused for approval ---")
print(result_1["__interrupt__"][0].value)

# Investment Committee Memo -- Cottonwood Solar

**Question:** How is the project performing against underwriting, and are there any open technical concerns?

## Document findings
- Cottonwood Solar -- Independent Engineer Report
 (Excerpt)
Cottonwood Solar is a 150 MW single-axis tracking PV facility in ERCOT West Texas, commercially
operational since 2022. Based on twelve months of measured irradiance and generation...
- Two unplanned inverter outages occurred in the first quarter of 2026, reducing quarterly output by
approximately 1.2%. Root cause was traced to a firmware defect in a batch of central inverters; the
vendor has issued a patch and the...

## Portfolio data
- 2025: revenue $19.4M (YoY +6.6%), EBITDA $14.8M
- Risk (Medium): Two unplanned inverter outages in Q1 2026 caused a temporary output shortfall.
- Offtake: Texas municipal aggregator -- fixed ($29.5/MWh)

--- QA review ---
[offline-fallback] Every number in the draft traces back to a research note.

--- Paused for a

In [17]:
# --- Investment Committee approves ------------------------------------------
from langgraph.types import Command

final_1 = await graph.ainvoke(Command(resume={"approved": True}), config_1)
print(final_1["final"])

# Investment Committee Memo -- Cottonwood Solar

**Question:** How is the project performing against underwriting, and are there any open technical concerns?

## Document findings
- Cottonwood Solar -- Independent Engineer Report
 (Excerpt)
Cottonwood Solar is a 150 MW single-axis tracking PV facility in ERCOT West Texas, commercially
operational since 2022. Based on twelve months of measured irradiance and generation...
- Two unplanned inverter outages occurred in the first quarter of 2026, reducing quarterly output by
approximately 1.2%. Root cause was traced to a firmware defect in a batch of central inverters; the
vendor has issued a patch and the...

## Portfolio data
- 2025: revenue $19.4M (YoY +6.6%), EBITDA $14.8M
- Risk (Medium): Two unplanned inverter outages in Q1 2026 caused a temporary output shortfall.
- Offtake: Texas municipal aggregator -- fixed ($29.5/MWh)

## QA Review
[offline-fallback] Every number in the draft traces back to a research note.

**Status: Approved fo

In [18]:
# --- Demo run 2: Blue Mesa Solar -- same graph, this time IC sends it back --
config_2 = {"configurable": {"thread_id": "bluemesa-2026-07"}}

result_2 = await graph.ainvoke(
    {
        "question": "What are the biggest risks to underwriting this asset going forward?",
        "project": "Blue Mesa Solar",
    },
    config_2,
)

print(result_2["draft"])
print("\n--- QA review ---")
print(result_2["qa_notes"])

# Investment Committee Memo -- Blue Mesa Solar

**Question:** What are the biggest risks to underwriting this asset going forward?

## Document findings
- interconnection node and the broader regional hub price has widened during several high-demand
periods.
Degradation is tracking marginally ahead of the Independent Engineer's original 0.5% per year curve,
which is a secondary contributor...
- Blue Mesa Solar -- Portfolio Risk Narrative
Blue Mesa Solar is a 120 MW solar facility in New Mexico, commercially operational since 2021. Its
original ten-year power purchase agreement expired in 2025, and the project has since operated...

## Portfolio data
- 2025: revenue $10.5M (YoY -11.0%), EBITDA $6.9M
- Risk (High): Operating fully merchant since PPA expiry; revenue volatility has increased.
- Offtake: wholesale market (no offtake counterparty) -- merchant (no fixed price)

--- QA review ---
[offline-fallback] Every number in the draft traces back to a research note.


In [19]:
# --- This time the committee sends it back for more work --------------------
final_2 = await graph.ainvoke(Command(resume={"approved": False}), config_2)
print(final_2["final"])

Memo drafted but not approved -- returned to the deal team for further work.


## Step 7 — Persistence that survives a restart

`MemorySaver` above is thread-scoped but lives in this process's RAM — restart the kernel and
every paused approval is gone. A real approval queue needs to survive that. LangGraph's SQLite
checkpointer persists to a file instead.

The next cell shows the first, natural thing to try, and why it fails.

In [20]:
# --- First attempt: the synchronous SqliteSaver -----------------------------
from langgraph.checkpoint.sqlite import SqliteSaver

config_3 = {"configurable": {"thread_id": "redtail-2026-07"}}

try:
    with SqliteSaver.from_conn_string(str(CHECKPOINT_DB)) as checkpointer:
        graph_sync_attempt = build_supervisor_graph(checkpointer)
        # This raises: SqliteSaver is a SYNCHRONOUS checkpointer, but every research node in this
        # graph is async (they `await tool.ainvoke(...)`), so the graph has to run through
        # `ainvoke`. A sync checkpointer can't service an async graph run -- it doesn't implement
        # the async methods the loop calls internally.
        result_3 = await graph_sync_attempt.ainvoke(
            {"question": "What's the interconnection and offtake picture, and is there schedule risk?",
             "project": "Redtail Ridge Wind"},
            config_3,
        )
except NotImplementedError as e:
    print("Failed, as expected:")
    print(e)

Failed, as expected:
The SqliteSaver does not support async methods. Consider using AsyncSqliteSaver instead.
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver
Note: AsyncSqliteSaver requires the aiosqlite package to use.
Install with:
`pip install aiosqlite`
See https://langchain-ai.github.io/langgraph/reference/checkpoints/#langgraph.checkpoint.sqlite.aio.AsyncSqliteSaverfor more information.


In [21]:
# --- Fix: AsyncSqliteSaver (needs the aiosqlite package) --------------------
from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver

async with AsyncSqliteSaver.from_conn_string(str(CHECKPOINT_DB)) as checkpointer:
    graph_persisted = build_supervisor_graph(checkpointer)
    result_3 = await graph_persisted.ainvoke(
        {
            "question": "What's the interconnection and offtake picture, and is there schedule risk?",
            "project": "Redtail Ridge Wind",
        },
        config_3,
    )
    print(result_3["draft"])
    print(f"\nPaused for approval. The checkpoint is now on disk at {CHECKPOINT_DB}, not in this process's RAM.")

# Investment Committee Memo -- Redtail Ridge Wind

**Question:** What's the interconnection and offtake picture, and is there schedule risk?

## Document findings
- Redtail Ridge Wind -- Interconnection & Offtake
 Summary
Redtail Ridge Wind is a 180 MW wind project in the SPP footprint in Oklahoma, currently under
construction with a revised target commercial operation date of the first quarter of...
- previously targeted commercial operation date from the fourth quarter of 2026 to the first quarter of
2027. The construction budget is otherwise tracking approximately 2% under the approved
contingency.
A fifteen-year power purchase...

## Portfolio data
- 2027: revenue $22.0M, EBITDA $17.0M
- Risk (Medium): Turbine supply delay pushed the commercial operation date by one quarter.
- Offtake: regional utility -- fixed ($27.75/MWh)

Paused for approval. The checkpoint is now on disk at /home/claude/build/run/deliver/lab3/data/helios_checkpoints.sqlite, not in this process's RAM.


In [22]:
# --- Simulate reopening this notebook after a restart -----------------------
# A brand-new AsyncSqliteSaver instance, pointed at the same file, built with no reference to
# `graph_persisted` above -- as close to "a fresh kernel" as one notebook process can get. Only
# `config_3["configurable"]["thread_id"]` and the file on disk carry the state across.
async with AsyncSqliteSaver.from_conn_string(str(CHECKPOINT_DB)) as checkpointer:
    graph_resumed = build_supervisor_graph(checkpointer)
    final_3 = await graph_resumed.ainvoke(Command(resume={"approved": True}), config_3)
    print(final_3["final"])

# Investment Committee Memo -- Redtail Ridge Wind

**Question:** What's the interconnection and offtake picture, and is there schedule risk?

## Document findings
- Redtail Ridge Wind -- Interconnection & Offtake
 Summary
Redtail Ridge Wind is a 180 MW wind project in the SPP footprint in Oklahoma, currently under
construction with a revised target commercial operation date of the first quarter of...
- previously targeted commercial operation date from the fourth quarter of 2026 to the first quarter of
2027. The construction budget is otherwise tracking approximately 2% under the approved
contingency.
A fifteen-year power purchase...

## Portfolio data
- 2027: revenue $22.0M, EBITDA $17.0M
- Risk (Medium): Turbine supply delay pushed the commercial operation date by one quarter.
- Offtake: regional utility -- fixed ($27.75/MWh)

## QA Review
[offline-fallback] Every number in the draft traces back to a research note.

**Status: Approved for Investment Committee.**


## Production notes

- **Docker.** Each server (`glossary_server.py`, `portfolio_server.py`, `docs_server.py`) is a
  small, independent process — one Dockerfile and container per server is the natural deployment
  unit, with `data/` and `mcp_servers/`-equivalent volumes mounted in.
- **Remote transport.** stdio only works for a locally-spawned subprocess. A remote MCP server
  (shared across analysts, not spawned per notebook) uses **Streamable HTTP** instead, with a
  bearer token in the connection config — `MultiServerMCPClient` accepts an HTTP entry alongside
  stdio ones, so the graph code above wouldn't change, just the server config dict.
- **Connection lifecycle.** Every call in this notebook spins up a fresh subprocess or HTTP
  request. That's fine for a lab; a production host would keep long-lived sessions (or a small
  pool) per server instead of paying a cold-start cost on every tool call.
- **Observability.** Nothing in this notebook talks to LangSmith directly — set
  `LANGSMITH_API_KEY` and `LANGCHAIN_TRACING_V2=true` as environment variables before building the
  graph, and every node above (research, writer, QA) is traced automatically. No code changes.
- **The approval gate is the real production boundary.** Automated QA here is a genuine,
  deterministic check (do the draft's numbers trace back to the research notes?) plus an optional
  LLM review — and it still isn't a substitute for the human sign-off. That's the point of keeping
  `approval` as a real `interrupt()`, not a rubber stamp.

## Verification

| Item | Status |
|---|---|
| Portfolio DB + 4 generated PDFs | Sandbox-verified |
| Embeddings (live HF vs. offline hashing fallback) | Sandbox-verified using `[offline-fallback]` (no route to huggingface.co here) |
| Chroma ingestion + project-scoped `search_documents` | Sandbox-verified |
| Three servers (`glossary`, `portfolio`, `docs`) via `MultiServerMCPClient` | Sandbox-verified |
| Supervisor graph: fan-out / fan-in / writer / QA | Sandbox-verified, `[offline-fallback]` text paths (no `GROQ_API_KEY` here) |
| Human approval via `interrupt()` / `Command(resume=...)` — both approve and reject | Sandbox-verified |
| `SqliteSaver` + async graph — the failure | Sandbox-verified (raises `NotImplementedError`, caught and shown) |
| `AsyncSqliteSaver` — true cross-session persistence | Sandbox-verified across independent context-manager instances |
| Live `ChatGroq` reasoning in each node | Written against the current API; needs a real `GROQ_API_KEY` to execute live |
| Remote MCP over Streamable HTTP + bearer tokens | Described in production notes; not exercised here (this lab only runs local stdio servers) |

## Edit points

- Add a fourth research node (e.g. a `web_researcher` hitting a news/regulatory search API) to
  `fanout`'s `Send` list and to `SupervisorState` — the `operator.add` reducer on `research_notes`
  means you don't need to touch `gather`, `writer`, or `qa` at all.
- Add a project to `SEED_PROJECTS` / `SEED_ECONOMICS` / `SEED_RISKS` / `SEED_OFFTAKE` in the
  portfolio-DB cell, generate a matching PDF, and add it to `PROJECT_TO_FILE` in
  `mcp_servers/docs_server.py`.
- Swap the QA node's deterministic numeric check for something stricter — e.g. flag any dollar
  figure in the draft that doesn't match a note to within rounding, not just an exact string match.
- Point the whole notebook at real infrastructure: set `GROQ_API_KEY` for live writer/QA
  reasoning, and `pip install langchain-huggingface sentence-transformers` plus internet access
  for real embeddings — no other code changes needed either way.

This closes the three-notebook arc: Lab 1 built one MCP server and connected it by hand; Lab 2
put a database behind MCP with a safety guard; this notebook connected three servers at once
behind a LangGraph supervisor with real parallel research, a genuine QA check, a human approval
gate, and checkpointing that survives a restart.